In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
import os
import re
import json
import pathlib
import tiktoken
import numpy as np
from openai import AzureOpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Process all PDF files in the /pdf folder and convert to markdown files in the /md folder
from markitdown import MarkItDown

# Get the current directory (using pathlib instead of __file__ which doesn't work in notebooks)
current_dir = str(pathlib.Path().absolute())

# Define input and output directories
pdf_dir = os.path.join(current_dir, "pdf")
faq_dir = os.path.join(current_dir, "faq")
text_dir = os.path.join(current_dir, "text")

# Format md files to be more readable
endpoint = os.getenv("endpoint")
openai_api_key = os.getenv("openai_api_key")

gpt4omini_model = "gpt-4o-mini"
embedding_model = "text-embedding-3-large"

# Initialize Azure OpenAI Service client with key-based authentication    
client = AzureOpenAI(
    azure_endpoint=endpoint,  
    api_key=openai_api_key,
    api_version="2025-01-01-preview",
)

# Get tokens length of the text
def get_tokens_length(text, encoding_name="cl100k_base"):
    """
    Returns the number of tokens in the text using the specified encoding.

    Args:
        text (str): The input text to be tokenized.
        encoding_name (str): The name of the tokenizer encoding to use.
    
    Returns:
        int: Number of tokens in the text.
    """
    # Get the encoding
    encoding = tiktoken.get_encoding(encoding_name)
    
    # Encode the text and return its length
    return len(encoding.encode(text))


def cosine_similarity(vec1, vec2):
    """
    Calculate the cosine similarity between two vectors.

    Args:
        vec1 (array-like): First vector.
        vec2 (array-like): Second vector.

    Returns:
        float: Cosine similarity score between -1 and 1.

    Raises:
        ValueError: If the vectors are not the same shape or if one vector is zero.
    """
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    
    if vec1.shape != vec2.shape:
        raise ValueError("Vectors must have the same dimensions.")
    
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    
    if norm1 == 0 or norm2 == 0:
        raise ValueError("One of the vectors is zero and cannot be normalized.")
    
    return np.dot(vec1, vec2) / (norm1 * norm2)


/workspaces/tech-blogs/venv/lib/python3.12/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


# <span style="color:yellow;">DEMO-1A: FAQ Text to JSON</span>

https://www.bsp.gov.ph/PaymentAndSettlement/FAQ_OPS_Registration.pdf

### System:
Using FAQ document below, return a json object containing: question, answer

Document:
### User:
Frequently Asked Questions (FAQs)
on Registration of Operators of Payment Systems
RA No. 11127 (The NPSA) and BSP Circular No. 1049
What is the National Payment Systems Act (NPSA)?
The NPSA is a landmark legislation that supports the performance by the Bangko Sentral of its
mandate relating to the third pillar of central banking – the maintenance of a safe, efficient, and
reliable payment and settlement systems.1
What are the objectives of the NPSA?
As the first comprehensive legal and regulatory framework governing payment systems in the
Philippines, the NPSA supports the twin objectives of maintaining safe, secure, efficient and
reliable operations of payment systems that is necessary to control systemic risk and of providing
an environment conducive to the sustainable growth of the economy.
What is BSP Circular No. 1049 about? What is its objective in relation to the first phase of
implementation of the NPSA?
Circular No. 1049 provides the rules and regulations on the registration of operators of payment
systems (OPS). It is the first phase of the phased-in implementation of the NPSA that prioritizes
the creation of a baseline inventory of all OPS. This is required under Section 10 of the NPSA
which provides that all OPS shall register with the Bangko Sentral. The inventory will be used as
inputs for the crafting of specific criteria in designating payment systems, as well as for the
oversight rules to be applied to such systems and its participants.
Circular No. 1049 provides descriptive examples, albeit not exhaustive, of activities to sufficiently
guide the stakeholders in determining which persons are required to register with the Bangko
Sentral as an OPS under the NPSA.2
In keeping with the thrust of the Bangko Sentral to promote efficiency and ease of doing
business, Circular No. 1049 contains a simplified registration process and streamlined
documentary requirements.3

# <span style="color:yellow;">LAB-1A: Convert PDF to Markdown Text</span>

In [3]:
# Initialize the MarkItDown converter
md = MarkItDown()

# Get a list of all PDF files in the pdf folder
pdf_files = [f for f in os.listdir(pdf_dir) if f.lower().endswith('.pdf')]

def pdf_to_markdown():
    # Process each PDF file
    for pdf_file in pdf_files:
        try:
            # Create full path for input file
            input_path = os.path.join(pdf_dir, pdf_file)
            
            # Create output filename (replace .pdf extension with .md)
            output_filename = os.path.splitext(pdf_file)[0] + '.md'
            output_path = os.path.join(text_dir, output_filename)
            
            print(f"Converting: {pdf_file}")
            
            # Convert the PDF file to markdown
            result = md.convert(input_path)
            content = result.text_content
            
            # Write the content to the markdown file
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(content)
            
            print(f"✓ Saved to: {output_filename}")
        except Exception as e:
            print(f"Error processing {pdf_file}: {str(e)}")

pdf_to_markdown()
print(f"\nConversion complete! {len(pdf_files)} PDF files processed.")
print(f"Markdown files saved to: {text_dir}")

Converting: EconomicOutlook_February2025.pdf


Cannot set gray non-stroke color because /'P0' is an invalid float value
Cannot set gray non-stroke color because /'P1' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P3' is an invalid float value


✓ Saved to: EconomicOutlook_February2025.md
Converting: FAQ_OPS_Registration.pdf
✓ Saved to: FAQ_OPS_Registration.md

Conversion complete! 2 PDF files processed.
Markdown files saved to: /workspaces/tech-blogs/notebooks/text


# <span style="color:yellow;">DEMO-1B: 🔢Token Counter</span>
https://platform.openai.com/tokenizer

In [3]:
file_path = os.path.join(text_dir, "EconomicOutlook_February2025.md")
print(f"File path: {file_path}")

with open(file_path, 'r', encoding='utf-8') as f:
    kb = f.read()

token_count = get_tokens_length(kb)
print(f"Tokens: {token_count}")

File path: c:\Users\robertrita\Workspace\CODES\_Mine\Chat-With-Data\notebooks\text\EconomicOutlook_February2025.md
Tokens: 5347
Tokens: 5347


# <span style="color:yellow;">LAB-1B: Convert Text to KB Topics</span>

In [2]:
#Prepare the chat prompt
system_topic = """
Using the input document below, return a JSON object separated by topic with the following fields: topic, content
- Make sure to faithfully retain all the document details. Do not remove to add any new information.

- Sample JSON output structure:
{
  "topics": [
    {  
      "topic": "<topic>",
      "content": "<content>"
    },
    ...
}

Document:
"""

# Get a list of all Markdown files in the md folder
md_files = [f for f in os.listdir(text_dir) if f.lower().endswith('.md')]

def markdown_to_topic():
    for md_file in md_files:
        try:
            if md_file not in ["EconomicOutlook_February2025.md"]:
                continue

            # Create full path for input file
            input_path = os.path.join(text_dir, md_file)
            print(f"Processing: {input_path}")
            
            with open(input_path, 'r', encoding='utf-8') as f:
                content = f.read()

            # Loop through the content and split it into pages
            pages = re.split(r'\f', content.strip())
            # Remove any empty pages
            pages = [page.strip() for page in pages if page.strip()]
            # Print the number of pages found
            print(f"Found {len(pages)} pages in {md_file}.")
            # Combine pages by 2 with 1 overlap
            pages = [pages[i] + "\n\n" + pages[i+1] for i in range(len(pages)-1)]
            contents = []

            # Loop each page and print
            for i, page in enumerate(pages):
                print(f"Page {i+1}:")
                # print(page)
                print("-" * 40)

                messages = [
                    {"role": "system", "content": system_topic},
                    {"role": "user", "content": page},
                ]

                # Generate the completion  
                completion = client.chat.completions.create(
                    model=gpt4omini_model,
                    messages=messages,
                    temperature=0,
                    top_p=1,
                    response_format={ "type": "json_object" },
                )

                data = json.loads(completion.choices[0].message.content)
                # Loop through the data topics
                for topic in data['topics']:
                    topic_name = topic['topic']
                    topic_content = topic['content']

                    response1 = client.embeddings.create(
                        input=f"{topic_name}\n\n{topic_content}",
                        model=embedding_model
                    )

                    found = False
                    # Loop through the contents
                    for item in contents:
                        # Calculate cosine similarity
                        similarity = cosine_similarity(response1.data[0].embedding, item['vector'])
                        # Print similarity
                        # print(f"Similarity: {similarity} - {topic_name}")

                        # Check if similarity is above the threshold
                        if float(similarity) > 0.95:
                            found = True

                    # Append the content to the list
                    if not found:
                        contents.append({
                            "topic": topic_name,
                            "content": topic_content,
                            "vector": response1.data[0].embedding,
                        })

            # print(contents)
            # break

            # Save the structured data to a JSON file
            output_filename = os.path.splitext(md_file)[0]
            output_path = os.path.join(faq_dir, output_filename + '.json')

            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(contents, f, indent=4)
            print(f"Knowledge base saved to: {output_path}")

        except Exception as e:
            print(f"Error processing {md_file}: {str(e)}")


markdown_to_topic()

Processing: c:\Users\robertrita\Workspace\CODES\_Mine\Chat-With-Data\notebooks\text\EconomicOutlook_February2025.md
Found 10 pages in EconomicOutlook_February2025.md.
Page 1:
----------------------------------------
Page 2:
----------------------------------------
Page 3:
----------------------------------------
Page 4:
----------------------------------------
Page 5:
----------------------------------------
Page 6:
----------------------------------------
Page 7:
----------------------------------------
Page 8:
----------------------------------------
Page 9:
----------------------------------------
Knowledge base saved to: c:\Users\robertrita\Workspace\CODES\_Mine\Chat-With-Data\notebooks\faq\EconomicOutlook_February2025.json


# <span style="color:yellow;">LAB-1C: Convert Text to KB FAQ</span>

In [6]:
#Prepare the chat prompt
system_faq = """
Using FAQ document below, return a JSON object with the following fields: question, answer
- Make sure to faithfully retain all the FAQ details. Do not remove to add any new information.
- The output must be a valid JSON object and works with Python json.loads().

- Sample JSON output structure:
{
  "faqs": [
    {  
      "question": "<question>",
      "answer": "<answer>"
    },
    ...
}

Document:
"""

# Get a list of all Markdown files in the md folder
md_files = [f for f in os.listdir(text_dir) if f.lower().endswith('.md')]

def markdown_to_faq():
    for md_file in md_files:
        try:
            if md_file not in ["FAQ_OPS_Registration.md"]:
                continue

            # Create full path for input file
            input_path = os.path.join(text_dir, md_file)
            print(f"Processing: {input_path}")
            
            with open(input_path, 'r', encoding='utf-8') as f:
                content = f.read()

            # Loop through the content and split it into pages
            pages = re.split(r'\f', content.strip())
            # Remove any empty pages
            pages = [page.strip() for page in pages if page.strip()]
            # Print the number of pages found
            print(f"Found {len(pages)} pages in {md_file}.")
            # Combine pages by 2 with 1 overlap
            pages = [pages[i] + "\n\n" + pages[i+1] for i in range(len(pages)-1)]
            contents = []

            # Loop each page and print
            for i, page in enumerate(pages):
                print(f"Page {i+1}:")
                # print(page)
                print("-" * 40)

                messages = [
                    {"role": "system", "content": system_faq},
                    {"role": "user", "content": page},
                ]

                # Generate the completion  
                completion = client.chat.completions.create(
                    model=gpt4omini_model,
                    messages=messages,
                    temperature=0,
                    top_p=1,
                    response_format={ "type": "json_object" },
                )

                raw = completion.choices[0].message.content
                # Print usage
                print("completion.usage: ", completion.usage.completion_tokens)

                # strip code fences and common noise
                raw = re.sub(r"```json", "", raw)
                raw = raw.replace("```", "").strip()
                try:
                    data = json.loads(raw)
                except json.JSONDecodeError as e:
                    print(f"❌ JSON parse error: {e}")
                    print("Raw response was:\n", raw)
                    continue

                for qa in data.get('faqs', []):
                    q, a = qa['question'], qa['answer']
                    vec = client.embeddings.create(input=f"{q}\n\n{a}", model=embedding_model).data[0].embedding

                    if not any(cosine_similarity(vec, item['vector']) > 0.95 for item in contents):
                        contents.append({"topic":q,"content":a,"vector":vec})
            # print(contents)
            # break

            # Save the structured data to a JSON file
            output_path = os.path.join(faq_dir, os.path.splitext(md_file)[0]+'.json')

            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(contents, f, indent=4)
            print(f"Knowledge base saved to: {output_path}")

        except Exception as e:
            print(f"Error processing {md_file}: {str(e)}")


markdown_to_faq()

Processing: c:\Users\robertrita\Workspace\CODES\_Mine\Chat-With-Data\notebooks\text\FAQ_OPS_Registration.md
Found 9 pages in FAQ_OPS_Registration.md.
Page 1:
----------------------------------------
completion.usage:  923
Page 2:
----------------------------------------
completion.usage:  1072
Page 3:
----------------------------------------
completion.usage:  939
Page 4:
----------------------------------------
completion.usage:  1004
Page 5:
----------------------------------------
completion.usage:  1068
Page 6:
----------------------------------------
completion.usage:  1023
Page 7:
----------------------------------------
completion.usage:  956
Page 8:
----------------------------------------
completion.usage:  704
Knowledge base saved to: c:\Users\robertrita\Workspace\CODES\_Mine\Chat-With-Data\notebooks\faq\FAQ_OPS_Registration.json
